<a href="https://colab.research.google.com/github/Azidian/Inteligencia_Artificial/blob/main/Componente_practico/Lecture_04/03_lab_warehouse_mdp_estudiantes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0, 0)
        self.walls = {
            (0, 3), (1, 1), (2, 4), (4,2)
        }
        self.slippery_states = {
            (1, 2), (2, 1), (3, 3)
        }

        self.terminal_states = {
            (0, 5): 10.0,
            (2, 2): 2.0,
            (3, 5): -10.0
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve [(next_state, probability), ...]
        para T(s,a,s') = P(s' | s,a).
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Determinar las probabilidades de movimiento según el tipo de piso
        if state in self.slippery_states:
            p_intended = 0.60
            p_perpendicular = 0.20
        else:
            p_intended = 0.90
            p_perpendicular = 0.05

        # Calcular direcciones perpendiculares para las desviaciones
        perp1 = (action[1], -action[0])
        perp2 = (-action[1], action[0])

        outcomes = [
            (action, p_intended),
            (perp1, p_perpendicular),
            (perp2, p_perpendicular),
        ]

        transitions = []
        for move, prob in outcomes:
            next_state = (state[0] + move[0], state[1] + move[1])

            # Si choca con un borde o pared, se queda en el mismo lugar
            if not self.is_valid_state(next_state):
                next_state = state

            transitions.append((next_state, prob))

        # Agrupar probabilidades si múltiples acciones llevan al mismo estado (ej. chocar contra la pared)
        grouped_transitions = {}
        for ns, p in transitions:
            grouped_transitions[ns] = grouped_transitions.get(ns, 0) + p

        return list(grouped_transitions.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # Suma de T(s,a,s') * V(s') para todos los posibles estados siguientes
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )

def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # Inicializamos todos los estados en 0
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(max_iter):
        biggest_change = 0.0
        V_new = V.copy()

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                # Ecuación de Bellman: R(s) + gamma * max_a expected_next_value
                V_new[state] = grid.get_reward(state) + grid.gamma * max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )

            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))

        V = V_new

        # Criterio de parada
        if biggest_change < threshold:
            break

    return V, iteration + 1

def extract_policy(grid, V):
    # Extraemos la mejor acción argmax_a para cada estado no terminal
    policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            continue

        # Encontramos la acción que maximice el valor esperado futuro
        best_action = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
        policy[state] = best_action

    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # Inicializamos todos los estados en 0
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                # Ecuación de Bellman sin MAX, porque la acción ya la dicta la política
                action = policy[state]
                V_new[state] = grid.get_reward(state) + grid.gamma * expected_next_value(grid, state, action, V)

            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))

        V = V_new
        if biggest_change < threshold:
            break

    return V

def policy_improvement(grid, V):
    new_policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            continue

        # Encontramos la acción que maximice el valor futuro esperado
        best_action = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
        new_policy[state] = best_action

    return new_policy

def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. Política inicial arbitraria (ej. moverse hacia ARRIBA en todos lados)
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []

    for iteration in range(max_iter):
        # 2. Evaluación
        V = policy_evaluation(grid, policy, threshold)

        # 3. Mejora
        new_policy = policy_improvement(grid, V)

        # Registramos cuántas acciones cambiaron para el historial
        cambios = sum(1 for s in policy if policy[s] != new_policy[s])
        history.append(f"Iteración {iteration + 1}: {cambios} cambios")

        # 4. Repetir hasta estabilidad
        if new_policy == policy:
            break

        policy = new_policy

    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: ['Iteración 1: 16 cambios', 'Iteración 2: 5 cambios', 'Iteración 3: 1 cambios', 'Iteración 4: 0 cambios']

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?

**Respuesta:** Al fijarse en la política nos damos cuenta que ambas políticas lo dirigen hacia la estación de carga, porque está más cerca y tiene menos penalización por los pasos. Matemátiamente determina que la resta sucesiva de -1 por caminar hasta allá destruye esos 10 puntos, haciendo que el +2 sea la opción que deja mayor saldo a favor.

2. ¿Por qué una recompensa menor podría ser óptima?

**Respuesta:** Eso se debe al costo de vida que tiene el robot. Cada paso le quita un punto, eso evita que se quede dando vueltas o no haga nada.
Teniendo en cuenta que llegar a la recompensa de +10 le toma mucho recorrido y le quita muchos puntos de vida prefiere el premio rápido y cercano.

3. ¿En qué estados el piso resbaloso cambia la decisión?

**Respuesta:** En los estados amarillos que están inmediatamente al lado o cerca de los peligros de -3 o -10. Como el piso resbaloso tiene un 40% de probabilidad de desvío lateral, el robot cambia su decisión para evitar transitar por esas baldosas si están junto a un riesgo, prefiriendo dar un rodeo largo por el piso normal (gris) donde tiene más control (90% de éxito).


4. ¿Qué papel cumple el costo por paso `-1`?

**Respuesta:** Obliga al robot a no perder tiempo, porque si no tuviera costo el vivir podría quedarse dando vueltas y jamás llegar a la meta que en este caso es la entrega. Otra opción sería que busque la opción más segura sin preocuparse de la demora que esta pueda llevarle. El objetivo de ponerle un costo es que priorice ciertos trayectos especificos con el objetivo de alivianar su penalización de vida, evitanto los peligros inminentes.

5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

**Respuesta:** Porque el almacen tiene distintas zonas, no todas las casillas son iguales, si asumimos que todas las probabilidades son homogéneas, estaríamos ignorando la estructura del mapa donde hay casillas que tienen piso resbaloso que cambian las probabilidades de tener un deslizamiento en 60% éxito, 40% desvío.

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

**Predicción:** Si cambiamos el costo de vida a -0.1 en vez de -1, es decir, caminar es mucho más barato ahora, el robot estará mucho más dispuesto a caminar trayectos largos.
Por lo tanto, el robot ya no irá a la recompensa pequeña de +2, sino que desde el inicio irá hacia la entrega +10, porque los pasos que necesita para llegar allá ya no superan la ganancia.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

**Predicción:** Si el piso resbaloso se vuelve mucho peor (solo 40% de éxito de ir hacia donde realmente queremos y 60% de desviarnos hacia los lados), el robot se volverá muy precavido. Evitará a toda costa pasar por las baldosas amarillas que estén cerca de los castigos de `-3` o `-10`, incluso si eso significa dar vueltas o tomar rutas más largas

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

**Respuesta:** Sí, si cambiamos el factor de descuento a `gamma = 0.99` (casi 1), el agente va a valorar el futuro lejano casi igual que el presente. Como el premio distante de `+10` retiene gran parte de su valor incluso después de muchos pasos, el robot ya no se conforma con el `+2` cercano y decide emprender el camino largo hacia la entrega principal

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.

**Respuesta:** El robot cambia su preferencia de ir a la estación de carga `+2` para intentar llegar a la entrega `+10` aproximadamente cuando el `living_reward` se vuelve mayor o igual a **`-0.5`**.